# August 2026 verification · **02 — Silver: RTT**

I turn the 17 monthly full extracts (April 2025 - August 2026) into tables
with an asserted grain, on **the same measure as Two Maps**:

* `RTT Part Description == "Incomplete Pathways"`, plus `"Incomplete Pathways with DTA"` kept as a separate part;
* commissioner `NONC` rows dropped, and the amount dropped reported, never silently lost;
* provider totals taken from treatment function `C_999` and asserted equal to the sum of the named functions;
* every measure recomputed from the weekly bands, not from any published percentage.

I write five tables to `data/silver/rtt/`:

| table | grain | what for |
|---|---|---|
| `rtt_provider.parquet` | period × part × provider | totals, within / over 18, over 52, over 104 |
| `rtt_england_bands.parquet` | period × part × week band | the England distribution, so I can rebuild the median |
| `rtt_provider_bands.parquet` | period × provider, 105 week-band columns | incomplete pathways only; lets Gold follow the near-52-week cohort on a constant panel |
| `rtt_nonc_dropped.csv` | period × part | what the NONC exclusion removed |
| `rtt_provider_presence.csv` | provider | which months each provider submitted. Mergers and non-submitters show up here |

Trust codes stay exactly as NHS England published them. The trust-merger
map is applied later, in Gold, because Gate A compares my rows with the
published workbook code for code.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, RTT_UNIDENTIFIED_COMMISSIONER, SILVER_DIR  # noqa: E402

OUT = SILVER_DIR / "rtt"
OUT.mkdir(parents=True, exist_ok=True)
PARTS = {"Incomplete Pathways": "incomplete", "Incomplete Pathways with DTA": "incomplete_dta"}
pd.set_option("display.width", 220)

## Step 1 — Which Bronze months do I have?

In [2]:
def rtt_zips() -> list[tuple[str, Path]]:
    """(period 'YYYY-MM', zip path) for every Bronze RTT extract, oldest first."""
    out = []
    for folder in sorted(BRONZE_DIR.glob("nhse_rtt_20??-??")):
        zips = list(folder.glob("*.zip"))
        assert len(zips) == 1, f"{folder}: expected one zip, found {len(zips)}"
        out.append((folder.name.replace("nhse_rtt_", ""), zips[0]))
    return out

months = rtt_zips()
print(len(months), "months:", months[0][0], "to", months[-1][0])
# I want an unbroken run: a missing month would quietly turn a two-month change
# into a "month-on-month" one.
expected = pd.period_range("2025-04", "2026-08", freq="M").strftime("%Y-%m").tolist()
assert [p for p, _ in months] == expected, "Bronze RTT months are not the unbroken April 2025 - August 2026 run"

17 months: 2025-04 to 2026-08


## Step 2 — Band columns

The wide headers run `Gt 00 To 01 Weeks SUM 1` … `Gt 103 To 104 Weeks SUM 1`,
then `Gt 104 Weeks SUM 1`. I parse each header's lower week rather than trust
column positions.

In [3]:
def band_lower_week(col: str) -> int:
    return int(col.split()[1])

def band_columns(cols) -> list[str]:
    bands = [c for c in cols if c.startswith("Gt ")]
    # 104 one-week bands plus the open-ended 104+ band
    assert len(bands) == 105, len(bands)
    assert [band_lower_week(c) for c in bands] == list(range(105)), "band headers out of order"
    return bands

## Step 3 — One month at a time

The definitions are the ones Two Maps used and the ones NHS England uses:

* **within 18 weeks** = bands with upper bound ≤ 18 weeks (weeks 0-18);
* **over 18 weeks** = total − within 18;
* **over 52 weeks** = bands with lower bound ≥ 52, including 104+;
* **over 104 weeks** = the 104+ band.

In [4]:
prov_bands = []   # filled by silver_month; provider x week band, incomplete pathways only


def silver_month(period: str, zip_path: Path):
    with zipfile.ZipFile(zip_path) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")]
        assert len(name) == 1, name
        with z.open(name[0]) as f:
            raw = pd.read_csv(f, low_memory=False)
    assert raw.Period.nunique() == 1, raw.Period.unique()
    bands = band_columns(raw.columns)
    lo = {c: band_lower_week(c) for c in bands}
    w18 = [c for c in bands if lo[c] + 1 <= 18]
    o52 = [c for c in bands if lo[c] >= 52]
    assert len(w18) == 18 and len(o52) == 53

    provs, eng_bands, dropped = [], [], []
    for desc, part in PARTS.items():
        d = raw[raw["RTT Part Description"] == desc].copy()
        assert len(d), f"{period}: no rows for {desc}"
        # The extract leaves 'Total' blank for incomplete pathways; 'Total All'
        # is populated and must equal the band sum exactly.
        d["total"] = d[bands].sum(axis=1)
        assert (d["total"] - d["Total All"]).abs().max() == 0, f"{period} {part}: band sum != Total All"

        nonc = d["Commissioner Org Code"].astype(str).str.startswith(RTT_UNIDENTIFIED_COMMISSIONER)
        is_tot = d["Treatment Function Code"] == "C_999"
        dropped.append({"period": period, "part": part,
                        "nonc_pathways": int(d.loc[nonc & is_tot, "total"].sum()),
                        "nonc_over_52": int(d.loc[nonc & is_tot, o52].sum().sum()),
                        "all_rows_pathways": int(d.loc[is_tot, "total"].sum())})
        d = d[~nonc]

        d["within_18"] = d[w18].sum(axis=1)
        d["over_52"] = d[o52].sum(axis=1)
        d["over_104"] = d["Gt 104 Weeks SUM 1"]
        keys = ["Provider Org Code", "Provider Org Name", "Provider Parent Org Code", "Provider Parent Name"]
        g = d.groupby(keys + ["Treatment Function Code"], as_index=False, dropna=False)[["total", "within_18", "over_52", "over_104"]].sum()

        # The C_999 'Total' row must equal the sum of the named treatment functions.
        tot = g[g["Treatment Function Code"] == "C_999"].set_index("Provider Org Code")
        named = g[g["Treatment Function Code"] != "C_999"].groupby("Provider Org Code").total.sum()
        gap = tot.total.sub(named, fill_value=0)
        assert (gap.abs() < 1).all(), f"{period} {part}: C_999 != sum of TFCs for {list(gap[gap.abs() >= 1].index)}"

        p = tot.reset_index().drop(columns="Treatment Function Code").rename(columns={
            "Provider Org Code": "provider_code", "Provider Org Name": "provider_name",
            "Provider Parent Org Code": "provider_parent_code", "Provider Parent Name": "provider_parent_name"})
        assert p.provider_code.is_unique, f"{period} {part}: grain violated"
        p["over_18"] = p.total - p.within_18
        p.insert(0, "part", part)
        p.insert(0, "period", period)
        provs.append(p)

        if part == "incomplete":
            # Provider x week band, so Gold can follow the near-52-week cohort on a
            # constant panel rather than on a provider set that changes month to month.
            pb = d.loc[d["Treatment Function Code"] == "C_999"].groupby("Provider Org Code")[bands].sum()
            pb.columns = [f"w{lo[c]:03d}" for c in bands]
            pb = pb.reset_index().rename(columns={"Provider Org Code": "provider_code"})
            pb.insert(0, "period", period)
            prov_bands.append(pb)
        b = d.loc[d["Treatment Function Code"] == "C_999", bands].sum()
        eng_bands.append(pd.DataFrame({"period": period, "part": part,
                                       "week_lower": [lo[c] for c in bands], "pathways": b.values.astype("int64")}))
    return pd.concat(provs), pd.concat(eng_bands), dropped

In [5]:
provs, bands_all, dropped_all = [], [], []
prov_bands.clear()   # so re-running this cell never double-counts
for period, zp in months:
    p, b, dr = silver_month(period, zp)
    provs.append(p); bands_all.append(b); dropped_all += dr
    inc = p[p.part == "incomplete"]
    print(f"{period}: providers {len(inc):>3} | pathways {inc.total.sum():>10,.0f} | "
          f">18wk {inc.over_18.sum():>9,.0f} | >52wk {inc.over_52.sum():>7,.0f} | NONC dropped {dr[0]['nonc_pathways']:>6,}")

rtt_provider = pd.concat(provs, ignore_index=True)
rtt_england_bands = pd.concat(bands_all, ignore_index=True)
nonc = pd.DataFrame(dropped_all)

2025-04: providers 469 | pathways  7,389,065 | >18wk 2,975,383 | >52wk 190,023 | NONC dropped 32,388
2025-05: providers 468 | pathways  7,355,950 | >18wk 2,873,874 | >52wk 196,839 | NONC dropped 32,327
2025-06: providers 471 | pathways  7,367,836 | >18wk 2,834,766 | >52wk 191,755 | NONC dropped 32,439
2025-07: providers 474 | pathways  7,299,023 | >18wk 2,830,554 | >52wk 189,416 | NONC dropped 35,263
2025-08: providers 476 | pathways  7,312,356 | >18wk 2,853,902 | >52wk 188,702 | NONC dropped 33,188
2025-09: providers 476 | pathways  7,296,893 | >18wk 2,791,209 | >52wk 178,532 | NONC dropped 32,681
2025-10: providers 478 | pathways  7,304,189 | >18wk 2,797,208 | >52wk 170,759 | NONC dropped 32,887
2025-11: providers 498 | pathways  7,159,010 | >18wk 2,746,148 | >52wk 154,121 | NONC dropped 32,300
2025-12: providers 504 | pathways  7,199,624 | >18wk 2,775,417 | >52wk 138,711 | NONC dropped 32,256
2026-01: providers 515 | pathways  7,155,958 | >18wk 2,760,000 | >52wk 133,860 | NONC dropp

## Step 4 — Internal consistency

In [6]:
assert not rtt_provider.duplicated(["period", "part", "provider_code"]).any()
# The England band table and the provider table are two routes to the same total.
chk = rtt_provider.groupby(["period", "part"]).total.sum() - rtt_england_bands.groupby(["period", "part"]).pathways.sum()
assert (chk == 0).all(), chk[chk != 0]
# DTA pathways are a subset of incomplete pathways, provider by provider.
w = rtt_provider.pivot_table(index=["period", "provider_code"], columns="part", values="total", aggfunc="sum").fillna(0)
assert (w.incomplete_dta <= w.incomplete).all(), "a provider has more DTA pathways than incomplete pathways"
print("grain, band / provider totals and DTA subset all hold")

grain, band / provider totals and DTA subset all hold


## Step 5 — Who submitted when?

A reported-provider series moves whenever a provider stops or starts
submitting, so I record presence now. This is a diagnostic, not a fix. Any
merger it shows up gets a dated entry in `TRUST_SUCCESSIONS`, with evidence.

In [7]:
inc = rtt_provider[rtt_provider.part == "incomplete"]
presence = inc.pivot_table(index=["provider_code"], columns="period", values="total", aggfunc="sum")
summary = pd.DataFrame({
    "provider_name": inc.drop_duplicates("provider_code", keep="last").set_index("provider_code").provider_name,
    "months_present": presence.notna().sum(axis=1),
    "first": presence.apply(lambda r: r.first_valid_index(), axis=1),
    "last": presence.apply(lambda r: r.last_valid_index(), axis=1),
    "max_pathways": presence.max(axis=1),
})
summary["gap_inside_run"] = presence.apply(
    lambda r: r.loc[r.first_valid_index():r.last_valid_index()].isna().any(), axis=1)
summary = summary.sort_values("max_pathways", ascending=False)
partial = summary[summary.months_present < len(months)]
print(f"{len(summary)} provider codes; {(summary.months_present == len(months)).sum()} present in all {len(months)} months")
print("NHS trusts (R-codes) not present in every month:")
partial[partial.index.str.startswith("R")]

552 provider codes; 448 present in all 17 months
NHS trusts (R-codes) not present in every month:


,provider_name,months_present,first,last,max_pathways,gap_inside_run
provider_code,,,,,,
RHQ,SHEFFIELD TEACHING HOSPITALS NHS FOUNDATION TRUST,3,2025-04,2025-06,92037.0,False
RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",16,2025-04,2026-08,72238.0,True
RK9,UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST,15,2025-04,2026-06,49961.0,False
RNS,NORTHAMPTON GENERAL HOSPITAL NHS TRUST,16,2025-04,2026-07,44903.0,False
RVJ,NORTH BRISTOL NHS TRUST,15,2025-04,2026-06,44445.0,False
RA9,TORBAY AND SOUTH DEVON NHS FOUNDATION TRUST,12,2025-04,2026-03,34622.0,False
R5J3Z,BORDER HEALTH ALLIANCE LIMITED,14,2025-07,2026-08,721.0,False
R6X7C,SPAMEDICA MACCLESFIELD COMMUNITY DIAGNOSTIC & ...,9,2025-12,2026-08,83.0,False
RW4,MERSEY CARE NHS FOUNDATION TRUST,12,2025-04,2026-03,55.0,False


## Step 6 — Write Silver

In [8]:
rtt_provider.to_parquet(OUT / "rtt_provider.parquet", index=False)
rtt_england_bands.to_parquet(OUT / "rtt_england_bands.parquet", index=False)
rtt_provider_bands = pd.concat(prov_bands, ignore_index=True)
# The band table must give back exactly the provider totals.
_t = rtt_provider_bands.set_index(["period", "provider_code"]).sum(axis=1)
_p = rtt_provider[rtt_provider.part == "incomplete"].set_index(["period", "provider_code"]).total
assert (_t.sort_index() == _p.sort_index()).all(), "provider band table does not rebuild provider totals"
rtt_provider_bands.to_parquet(OUT / "rtt_provider_bands.parquet", index=False)
nonc.to_csv(OUT / "rtt_nonc_dropped.csv", index=False)
summary.to_csv(OUT / "rtt_provider_presence.csv")
print("written:", sorted(f.name for f in OUT.iterdir()))

written: ['rtt_england_bands.parquet', 'rtt_nonc_dropped.csv', 'rtt_provider.parquet', 'rtt_provider_bands.parquet', 'rtt_provider_presence.csv']
